01. Import & Setup NLTK

In [1]:
import pandas as pd
import re
import nltk

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Load data ulang (karena file baru)
df = pd.read_csv('../data/SMS Spam Collection Dataset.csv', encoding='latin-1')
df = df[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'text'})

print("Data loaded:", df.shape)

Data loaded: (5572, 2)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


02. Preprocessing Function

In [2]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # 1. Lowercase
    text = text.lower()
    
    # 2. Hapus karakter non-alfabet (angka, simbol, tanda baca)
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 3. Tokenisasi (pecah jadi list kata)
    tokens = text.split()
    
    # 4. Hapus stopwords dan lakukan stemming
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]
    
    # 5. Gabung kembali jadi string
    return ' '.join(tokens)

# Test dulu dengan satu kalimat
contoh = "WINNER!! You have been selected to receive a FREE prize! Call 08001234567 NOW!"
print("Sebelum:", contoh)
print("Sesudah:", preprocess(contoh))

Sebelum: WINNER!! You have been selected to receive a FREE prize! Call 08001234567 NOW!
Sesudah: winner select receiv free prize call


03. Implementation

In [3]:
# Menerapkan preprocessing ke semua teks
df['text_clean'] = df['text'].apply(preprocess)

# Bandingkan sebelum vs sesudah
print("=== Perbandingan Sebelum vs Sesudah ===\n")
for i in [2, 8, 9]:  # index spam
    print(f"SEBELUM: {df['text'][i]}")
    print(f"SESUDAH: {df['text_clean'][i]}")
    print()

# Cek tidak ada yang kosong setelah preprocessing
print("Teks kosong setelah preprocessing:", (df['text_clean'] == '').sum())
print("\nSample hasil akhir:")
print(df[['label', 'text', 'text_clean']].head())

=== Perbandingan Sebelum vs Sesudah ===

SEBELUM: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
SESUDAH: free entri wkli comp win fa cup final tkt st may text fa receiv entri questionstd txt ratetc appli over

SEBELUM: WINNER!! As a valued network customer you have been selected to receivea å£900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.
SESUDAH: winner valu network custom select receivea prize reward claim call claim code kl valid hour

SEBELUM: Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030
SESUDAH: mobil month u r entitl updat latest colour mobil camera free call mobil updat co free

Teks kosong setelah preprocessing: 6

Sample hasil akhir:
  label                                               text  \
0   ham  Go until jurong point

04. Text Clean

In [4]:
# Hapus baris dengan teks kosong setelah preprocessing
df = df[df['text_clean'] != ''].reset_index(drop=True)

print("Shape setelah hapus teks kosong:", df.shape)
print("Distribusi label setelah cleaning:")
print(df['label'].value_counts())

# Simpan hasil preprocessing
df.to_csv('../data/data_clean.csv', index=False)
print("\nData clean tersimpan di data/data_clean.csv")

Shape setelah hapus teks kosong: (5566, 3)
Distribusi label setelah cleaning:
label
ham     4819
spam     747
Name: count, dtype: int64

Data clean tersimpan di data/data_clean.csv
